# 7회차 · 내 데이터로 모델 학습시키기(LoRA 샌드박스)

오늘은 모델의 **가중치를 직접 바꿈**. 내가 원하는 말투 · 형식을 정하고, 학습 데이터를 만들고, 설정을 바꿔 가며 여러 번 학습시켜 차이를 확인함.
마지막에는 6회차에 만든 내 문서 챗봇에 학습한 Adapter 를 얹어 **내 솔루션**을 완성함.

**전체 흐름부터.** 5회차와 6회차는 모델을 그대로 두고 입력을 바꿨음(지시문, 문서). 그 방법은 매번 지시문을 붙여야 하고 입력이 길어짐. 오늘은 반대로 **모델 쪽을 조금 고쳐서** 지시문 없이도 원하는 방식으로 답하게 만듦.

모델 전체(수십억 개의 숫자)를 다시 학습시키면 시간과 메모리가 너무 많이 듦. **LoRA** 는 원래 가중치(본체)는 얼려 두고, 옆에 작은 행렬(Adapter)만 붙여 그 부분만 학습하는 방법임. 두꺼운 교과서는 그대로 두고 얇은 보충 노트만 새로 쓰는 것과 비슷함. 그래서 1~3분이면 끝나고 결과 파일도 수십 MB 임.

`학습 목표(1단계)` → 학습 데이터 수백 건(3단계) → 학습(4단계) → 지시문 없이 물어 확인(5단계) → 설정을 바꿔 다시(6단계) → 문서 검색과 합침(9단계)

RAG 와 LoRA 는 하는 일이 다름. **무엇을 아는가(지식)** 는 문서를 붙이는 RAG 가, **어떻게 답하는가(말투 · 형식)** 는 LoRA 가 잘 바꿈. 9단계에서 숫자로 확인함.

- 학습은 한 번에 1~3분임. 설정을 바꿔 **세 번 이상** 학습해 봄
- 끝낼 때 맨 아래 '결과 묶기'를 실행해 zip 파일을 내려받음

| 단계 | 하는 일 | 필수 |
|---|---|---|
| 0 | 시작 : 6회차 솔루션 불러오기 | 필수 |
| 1 | 학습 목표 정하기 | 필수 |
| 2 | 학습 전 답 확인 | 필수 |
| 3 | 학습 데이터 만들기 | 필수 |
| 4 | 첫 학습 | 필수 |
| 5 | 전후 비교 | 필수 |
| 6 | 설정 실험 : 다시 학습하고 곡선 비교 | 필수 |
| 7 | 일부러 문제 만들기 | 선택 |
| 8 | 큰 GPU 써 보기(QLoRA · 큰 배치 · 다른 본체) | 선택 |
| 9 | 네 구성 비교 | 필수 |
| 10 | 내 솔루션 완성 · 저장 | 필수 |
| 11 | 판단 기록 · 결과 묶기 | 필수 |

선택 단계(7 · 8)는 **하나 이상** 함. 목표를 정하기 어려우면 맨 아래 '예시 과제'를 봄.

## 0. 시작
아래 두 셀을 차례로 실행함. 6회차에 저장한 솔루션(모델 · 문서 · 검색 설정)을 그대로 불러옴. 6회차 결과가 없으면 `lab.docs_panel()` 로 문서를 넣고 `lab.quiz_panel()` 로 확인 질문을 적은 뒤 이어 감.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(7)

In [ ]:
lab.load_solution()
lab.gpu()

## 1. 학습 목표 정하기
모델이 **어떻게 답하게 만들 것인지** 정함. 준비된 목표를 고르면 입력란이 채워지고, 그대로 쓰거나 고쳐 씀. 직접 새로 적어도 됨.
- **지시문** : 원하는 말투 · 형식 · 길이. 학습 데이터를 만들 때만 쓰고, 학습한 모델에는 주지 않음
- **표지 단어** : 목표대로 답했는지 판정하는 단어나 문장 끝 표현. 점수 계산에 씀
- **예시** : `질문 => 내가 원하는 답`. 내가 원하는 답의 모습 그대로 3건 이상 적음. 예시의 품질이 학습 결과를 좌우함

**왜 하나** : 학습은 "이런 질문에는 이렇게 답하라"는 본보기를 수백 번 보여 주는 일임. 본보기가 흐릿하면 결과도 흐릿함. 그리고 학습이 됐는지 숫자로 확인하려면 판정 기준이 필요한데, 표지 단어가 그 기준임(예 : 사극 말투라면 `하오`, `이오`). 표지 단어를 정할 수 없는 목표는 점수로 잴 수 없으므로 목표를 더 구체적으로 고침.

In [ ]:
lab.goal_panel()

## 2. 학습 전 답 확인
학습하기 전의 모델에게 지시문 없이 물어봄. 지금은 평범하게 답하는 것이 정상임. 같은 질문을 5단계에서 다시 물어 비교함.

**왜 하나** : 고치기 전의 모습을 봐 두지 않으면 학습으로 무엇이 바뀌었는지 말할 수 없음. 질문 두세 개를 정해 두고 답을 기억해 둠.

In [ ]:
lab.compare_panel()

## 3. 학습 데이터 만들기
학습에는 `질문 → 원하는 답` 쌍이 수백 건 필요함. 손으로 다 쓸 수 없으므로 지시문과 예시를 받은 **교사 모델**이 질문 수백 건에 목표대로 답하게 하고, 모델이 낸 답을 학습 데이터로 씀.

**왜 하나** : 지시문을 받으면 잘 답하는 모델(교사)의 답을 모아서, 지시문 없이도 그렇게 답하는 모델(학생)을 만드는 것임. 이때 교사와 학생은 같은 모델임. 지시문이 있을 때의 답을 지시문이 없을 때도 내도록 가르친다고 보면 됨.

| 바꿔 볼 것 | 값 | 볼 것 |
|---|---|---|
| 만들 건수 | 60 → 200 → 400 | 6단계에서 건수에 따라 말투 점수가 어떻게 달라지는가 |
| 내 문서 섞는 비율 | 0 → 0.4 → 0.7 | 9단계 '문서 + Adapter'의 말투 점수. 0 이면 문서를 붙였을 때 말투가 사라지기 쉬움 |
| 표지 단어 기준 | 켬 대 끔 | '기준에 못 미쳐 뺀 답'이 몇 건인가 |

- 진행 막대의 '한 번에 몇 건'이 GPU 메모리에 따라 정해짐. 메모리가 클수록 한꺼번에 많이 생성함
- 만들어진 표를 **꼭 읽어 봄**. 답이 목표와 다르면 1단계로 돌아가 지시문 · 예시를 고친 뒤 다시 만듦. 데이터가 틀리면 모델은 틀린 대로 배움

In [ ]:
lab.data_panel()

## 4. 첫 학습
기본 설정 그대로 '학습 시작'을 누름. Loss 곡선이 실시간으로 그려짐.

**곡선 읽는 법** : **Loss** 는 모델의 답이 학습 데이터의 답과 얼마나 다른지 나타내는 값임. 내려가면 학습 데이터를 닮아 가고 있다는 뜻임. 실선은 학습에 쓴 데이터의 Loss, 점선은 학습에 쓰지 않고 따로 떼어 둔 검증 데이터의 Loss 임. 실선은 계속 내려가는데 점선이 다시 올라가기 시작하면, 모델이 데이터를 이해한 것이 아니라 외우고 있다는 신호임(과적합).

- 표에서 **학습한 파라미터 비율**과 **Adapter 크기**를 확인함. 전체의 1% 도 안되는 숫자만 고쳤는데 답이 바뀜
- 학습 중에 계기판을 보면 메모리가 모델만 올렸을 때보다 늘어나 있음. 학습은 계산 중간값을 저장해야 해서 메모리를 더 씀

In [ ]:
lab.train_panel()

## 5. 전후 비교
2단계와 같은 질문, 그리고 학습 데이터에 없던 새 질문으로 물어봄. 표지 단어는 색으로 표시됨. 아래 두 번째 셀은 검증 질문 20건으로 **말투 점수**를 재서 막대로 그림.

**왜 하나** : 학습 데이터에 있던 질문에 잘 답하는 것은 당연함. **처음 보는 질문**에도 목표대로 답해야 학습이 된 것임. 그래서 말투 점수는 학습에 쓰지 않은 질문으로 잼.

In [ ]:
lab.compare_panel()

In [ ]:
lab.style_eval()

## 6. 설정 실험
4단계 조절판으로 돌아가 **설정 하나만** 바꿔 다시 학습함(이름은 비워 두면 v2, v3 … 로 붙음). 실행 전에 결과를 예측해 봄.

**설정이 뜻하는 것**
- **rank(r)** : Adapter 행렬의 크기. 크면 더 많은 것을 배울 수 있지만 파일이 커지고, 데이터가 적을 때는 외우기 쉬워짐
- **학습률** : 한 번에 얼마나 크게 고칠지. 너무 작으면 거의 안 변하고, 너무 크면 Loss 가 내려가지 않고 튐
- **에폭** : 같은 데이터를 몇 번 반복해 볼지. 많을수록 Loss 는 내려가지만 과적합 위험이 커짐
- **배치 크기** : 한 번에 몇 건을 묶어 계산할지. 크면 빠르지만 메모리를 더 씀

| 바꿔 볼 것 | 살펴볼 것 |
|---|---|
| rank 2 대 64 | 학습한 파라미터 수 · Adapter 크기 · 말투 점수 |
| 에폭 1 대 6 | Loss 가 어디까지 내려가는가 · 답이 학습 데이터를 그대로 외우는가 |
| 학습 건수 40 대 전부 | 데이터가 적어도 되는가 |
| 붙일 층 q · v 대 전체 | 학습 시간 · 점수 |

두 번 이상 더 학습한 뒤 아래 셀로 Loss 곡선을 한 그림에 함께 그리고 점수를 다시 잼.

In [ ]:
lab.loss_plot()
lab.style_eval()

## 7. 일부러 문제 만들기(선택)
잘못된 설정이 어떤 모습으로 나타나는지 직접 봄. 조절판에서 아래처럼 고르고 학습한 뒤 5단계 조절판으로 답을 읽어 봄.

**왜 하나** : 실제로 학습을 돌리다 보면 결과가 이상할 때가 훨씬 많음. 그때 곡선과 답의 모습만 보고 원인을 짐작할수 있어야 함. 문제를 일부러 만들어서 그 모습을 미리 봐 두는 것임.

- **학습률 0.01** : Loss 가 내려가지 않거나 치솟음. 답이 깨지는지 확인함
- **에폭 12 · 학습 건수 20** : Loss 는 0 에 가까워지는데 새 질문에는 맞지 않는 답을 함(과적합)
- 1단계에서 예시를 일부러 틀리게 적고 데이터를 다시 만들면, 모델은 틀린 답을 그대로 배움

일부러 문제를 만든 학습은 최종 솔루션으로 고르지 않음. 이름을 `bad1` 처럼 붙여 두면 헷갈리지 않음.

## 8. 큰 GPU 써 보기(선택)
- **QLoRA** : 본체 자료형을 4bit 로 바꿔 학습함. 표의 '학습 메모리'와 걸린 시간, 말투 점수를 bf16 과 비교함. 본체를 4bit 로 줄여 올리면 메모리는 크게 줄고 속도는 조금 느려짐. 작은 GPU 에서 학습할 때 쓰는 방법임
- **큰 배치** : 배치를 32 · 64 로 키움. 한 번에 많이 보지만 갱신 횟수가 줄어듦. 점수가 어떻게 되는지 봄
- **다른 본체** : '학습할 본체'를 `Qwen3-8B` 로 바꿔 같은 데이터로 학습함. 학습한 Adapter 는 그 본체에서만 쓸 수 있으므로, 아래 셀로 본체를 올리고 비교함

**왜 하나** : 같은 학습이 GPU 크기에 따라 어떻게 달라지는지 보는 단계임. 96GB 에서는 원래 정밀도(bf16)로 큰 배치를 돌릴 수 있고, 24GB 에서는 4bit 와 작은 배치를 써야 함. 나중에 내 GPU 로 학습할 때 무엇을 줄여야 하는지 미리 알아 둠.

In [ ]:
lab.gpu()
# 다른 본체로 학습했으면: lab.load('Qwen/Qwen3-8B') 뒤 lab.compare('질문') 으로 비교하고, 끝나면 lab.use('A.X-4.0-Light') 로 돌아옴

## 9. 네 구성 비교
6회차에 적은 확인 질문 전체를 네 가지 구성으로 풀어 **내용 점수**(기대 단어)와 **말투 점수**(표지 단어)를 함께 봄.
- 학습 전 · 문서 붙임(RAG) · Adapter(LoRA) · 문서 + Adapter

**왜 하나** : 오늘 수업의 결론이 이 그림 한 장임. 내용 점수는 문서를 붙일 때 오르고, 말투 점수는 Adapter 를 쓸 때 오름. 둘을 같이 쓰면 둘 다 오르는지, 아니면 문서를 붙이는 순간 말투가 사라지는지 내 데이터로 확인함. 사라졌다면 3단계의 '내 문서 섞는 비율'을 올려 데이터를 다시 만들고 학습함.

In [ ]:
lab.four()

## 10. 내 솔루션 완성
가장 마음에 드는 Adapter 이름을 넣어 저장함. 저장한 뒤 대화 창에서 써 봄. 8회차부터는 저장한 솔루션을 불러 써서 서비스로 만듦.

In [ ]:
lab.runs()

In [ ]:
lab.save_solution(adapter='v1')     # 고른 Adapter 이름으로 바꿈

In [ ]:
lab.chat()

## 11. 판단 기록 · 결과 묶기
어떤 설정의 Adapter 를 골랐고 왜 골랐는지, 무엇이 기대와 달랐는지 적음. 근거가 되는 숫자를 두 개 이상 넣음(말투 점수, 학습 시간, Adapter 크기 등).
끝으로 결과 묶기를 실행하고 zip 파일을 내려받음.

In [ ]:
lab.note_panel()

In [ ]:
lab.pack()

---
### 예시 과제(목표를 정하지 못했을 때 하나를 고름)
1. **말투 챗봇** : 준비된 목표(사극 말투, 해적 선장)를 고르거나 좋아하는 캐릭터의 말투를 직접 적음. 지시문 없이 말투 점수 0.8 이상을 목표로 함
2. **세 줄 요약 모델** : 어떤 질문에도 정확히 세 줄로 답하게 함. 표지 단어 대신 형식이 지켜졌는지를 5단계 답에서 직접 확인함
3. **우리 동아리 상담원** : 6회차 문서 챗봇에 친절한 상담원 말투를 입힘. 9단계에서 '문서 + Adapter'의 내용 점수와 말투 점수가 둘 다 0.8 이상이 되게 함

### 도전 과제
- **가장 작은 Adapter** : 말투 점수 0.8 을 넘는 가장 작은 rank 와 가장 적은 학습 건수를 찾음
- **30초 학습** : 에폭 · 건수 · 배치를 조정해 학습 시간을 30초 안으로 줄이고도 점수를 유지함
- **말투 두 개** : 서로 다른 목표로 Adapter 를 두 개 학습하고 대화 창의 Adapter 선택 상자로 바꿔 가며 써 봄. 본체는 하나인데 Adapter 만 바꿔 끼우는 것임
- **문서를 붙여도 말투 유지** : '내 문서 섞는 비율' 0 과 0.6 으로 각각 학습해 9단계 결과를 비교함
- 조절판 없이 직접 부르기 : `lab.train(r=8, epochs=2)`, `lab.compare("질문")`, `lab.style_eval()`, `lab.four()`